
# Solution Key to Assignment on Neural Network


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score
)

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 5)


## Shared Example Dataset

In [2]:
in_file_name = "gene_expression_binary.csv"
df = pd.read_csv(in_file_name)

print(df.shape)
print(df.head())

(180, 301)
     Gene_1    Gene_2    Gene_3    Gene_4    Gene_5    Gene_6    Gene_7  \
0  0.165362  1.052481  0.250059  0.647174  0.999504  7.399953 -0.139929   
1 -0.780324 -0.108025  0.716442  0.435419 -0.609848 -0.242195 -0.439404   
2  0.231719 -1.680229 -0.565501 -0.042879 -0.494839  4.032141  0.764033   
3  0.491246  0.388259 -1.901445  1.927810  0.204609  4.682180  0.168620   
4 -2.448968  2.164372 -0.509130 -0.013008  1.309486 -2.768720 -0.479120   

     Gene_8    Gene_9   Gene_10  ...  Gene_292  Gene_293   Gene_294  Gene_295  \
0 -0.825100 -1.731810  0.490245  ...  1.835600  1.515755  -0.091484 -0.932527   
1  0.193077  2.019842  1.014541  ...  1.146299  0.978062 -12.977023 -0.012661   
2 -1.069174  0.868155  1.517235  ...  0.307717 -1.219386  17.469884 -1.112189   
3 -0.259217 -0.272536 -0.005169  ...  0.628027  0.454545  -6.172395 -1.510199   
4  0.468360 -1.277465 -0.972258  ...  1.002873 -0.736546  -0.641376  0.128546   

   Gene_296  Gene_297  Gene_298  Gene_299  Gene_300

In [3]:
X = df.drop(columns=["disease_label"]).values
feature_names = df.drop(columns=["disease_label"]).columns.tolist()

y = df["disease_label"].values


## Problem: Build a Neural Network


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

Train shape: (126, 300) (126,)
Test shape: (54, 300) (54,)



Feature scaling matters because neural networks usually train more stably when features are on similar scales.

Fit the scaler on the **training set only**, then apply it to validation and test data.


In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training means (first 3 variables):", X_train_scaled.mean(axis=0)[:3].round(4))
print("Scaled training stds (first 3 variables):", X_train_scaled.std(axis=0)[:3].round(4))

Scaled training means (first 3 variables): [ 0.  0. -0.]
Scaled training stds (first 3 variables): [1. 1. 1.]



## Problem 2 — Build a Simple Neural Network Classifier in Python

In `MLPClassifier`, binary probabilities are obtained with `predict_proba`.


In [6]:
mlp = MLPClassifier(
    hidden_layer_sizes=(16,),
    activation="relu",
    solver="adam",
    max_iter=500,
    random_state=42
)

mlp.fit(X_train_scaled, y_train)

test_proba = mlp.predict_proba(X_test_scaled)[:, 1]
val_pred = (test_proba >= 0.5).astype(int)

print("First 10 test probabilities:", np.round(test_proba[:10], 4))
print("First 10 test predictions:", val_pred[:10])
print("test accuracy:", round(accuracy_score(y_test, val_pred), 4))


First 10 test probabilities: [0.7238 0.9746 0.6464 0.0426 0.0974 0.0655 0.0286 0.9544 0.0412 0.0987]
First 10 test predictions: [1 1 1 0 0 0 0 1 0 0]
test accuracy: 0.7407



**Predicted probability** is a continuous confidence score from 0 to 1.  
**Predicted class** is obtained by thresholding that probability, often at 0.5.
